### **<h3 style="color:pink;"> RAG System — Week 10: Monitoring Dashboard**

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Introduction**</span>

</div>

This week we build monitoring for our RAG system:
- ✅ Track API performance metrics
- ✅ Monitor response times
- ✅ Detect data drift with Evidently
- ✅ Log everything to MLflow

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup & Imports**</span>

</div>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import requests
import json
import time
import pandas as pd
import numpy as np
import mlflow
from datetime import datetime

# MLflow setup
mlflow.set_tracking_uri("sqlite:///C:/Users/USER/Documents/RAG_Project/mlflow.db")
mlflow.set_experiment("RAG_Monitoring")

API_BASE = "http://127.0.0.1:8000"

print("✅ Imports successful!")
print(f"📍 API: {API_BASE}")

2026/04/01 22:06:53 INFO mlflow.tracking.fluent: Experiment with name 'RAG_Monitoring' does not exist. Creating a new experiment.


✅ Imports successful!
📍 API: http://127.0.0.1:8000


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Health Check & API Test**</span>

</div>

In [2]:
# Check API health
print("🔍 Checking API health...")
try:
    response = requests.get(f"{API_BASE}/health")
    health = response.json()
    print(f"✅ API Status: {health['status']}")
    print(f"✅ Model: {health['model']}")
    print(f"✅ Chunks indexed: {health['chunks_indexed']}")
except Exception as e:
    print(f"❌ API not running! Start it with:")
    print(f"   uvicorn src.serving.api:app --reload --port 8000")
    print(f"   Error: {e}")

🔍 Checking API health...
✅ API Status: healthy
✅ Model: llama-3.1-8b-instant
✅ Chunks indexed: 23562


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Performance Testing**</span>

</div>

Running 10 test queries to measure API performance!

In [3]:
# Test questions
test_questions = [
    "What are the liability rules for business entities?",
    "Who is authorized to require additional terms?",
    "What defines gross negligence in this context?",
    "What are the rules for nonprofit organizations?",
    "What happens when a volunteer causes injury?",
    "Who prescribes regulations for expenditures?",
    "What is the definition of a business entity?",
    "What conditions remove business liability?",
    "Who selects the chairperson of the Panel?",
    "What are the Secretary of Agriculture powers?"
]

print("⏳ Running 10 test queries...")
print("☕ This will take ~30 seconds\n")

results = []
for i, question in enumerate(test_questions):
    start = time.time()
    try:
        response = requests.post(
            f"{API_BASE}/query",
            json={"question": question, "top_k": 3}
        )
        data = response.json()
        elapsed = time.time() - start
        
        results.append({
            "question": question[:50],
            "status": "success",
            "response_time": round(elapsed, 2),
            "answer_length": len(data.get("answer", "")),
            "sources_count": len(data.get("sources", []))
        })
        print(f"✅ Q{i+1}: {elapsed:.2f}s — {data['answer'][:80]}...")
        
    except Exception as e:
        elapsed = time.time() - start
        results.append({
            "question": question[:50],
            "status": "error",
            "response_time": round(elapsed, 2),
            "answer_length": 0,
            "sources_count": 0
        })
        print(f"❌ Q{i+1}: Error — {e}")

print(f"\n✅ Done! {len(results)} queries completed")

⏳ Running 10 test queries...
☕ This will take ~30 seconds

✅ Q1: 6.88s — According to the provided context, the liability rules for business entities are...
✅ Q2: 0.67s — The Secretary of Agriculture is authorized to require additional terms....
✅ Q3: 0.58s — In this context, gross negligence is defined as "voluntary and conscious conduct...
✅ Q4: 0.83s — Based on the provided context, the rules for nonprofit organizations are as foll...
✅ Q5: 0.64s — Based on the provided context, when a volunteer causes injury, the protection fr...
✅ Q6: 0.53s — Based on the provided context, it appears that the implementing regulations are ...
✅ Q7: 0.69s — According to the provided context, a business entity is defined as:

"A firm, co...
✅ Q8: 0.64s — Based on the provided context, the conditions that remove business liability are...
✅ Q9: 0.58s — Based on the provided context, the chairperson of the Panel is selected by the m...
✅ Q10: 0.58s — Based on the provided context, the Secretary of Agric

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Performance Analysis**</span>

</div>

In [4]:
# Analyze performance
df = pd.DataFrame(results)

print("=" * 60)
print("📊 PERFORMANCE REPORT:")
print("=" * 60)
print(f"\n⏱️  Response Times:")
print(f"   Min      : {df['response_time'].min():.2f}s")
print(f"   Max      : {df['response_time'].max():.2f}s")
print(f"   Average  : {df['response_time'].mean():.2f}s")
print(f"   Median   : {df['response_time'].median():.2f}s")

print(f"\n✅ Success Rate:")
success = len(df[df['status'] == 'success'])
print(f"   {success}/{len(df)} queries successful ({success/len(df)*100:.0f}%)")

print(f"\n📝 Answer Quality:")
print(f"   Avg answer length : {df['answer_length'].mean():.0f} chars")
print(f"   Avg sources found : {df['sources_count'].mean():.1f}")

print(f"\n🏆 Performance Grade:")
avg_time = df['response_time'].mean()
if avg_time < 2:
    print(f"   ⚡ EXCELLENT! Average {avg_time:.2f}s")
elif avg_time < 5:
    print(f"   ✅ GOOD! Average {avg_time:.2f}s")
else:
    print(f"   ⚠️  SLOW! Average {avg_time:.2f}s")
print("=" * 60)

📊 PERFORMANCE REPORT:

⏱️  Response Times:
   Min      : 0.53s
   Max      : 6.88s
   Average  : 1.26s
   Median   : 0.64s

✅ Success Rate:
   10/10 queries successful (100%)

📝 Answer Quality:
   Avg answer length : 382 chars
   Avg sources found : 3.0

🏆 Performance Grade:
   ⚡ EXCELLENT! Average 1.26s


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Logging to MLflow**</span>

</div>

In [5]:
# Log monitoring results to MLflow
with mlflow.start_run(run_name="monitoring_baseline"):
    # Log parameters
    mlflow.log_param("api_url", API_BASE)
    mlflow.log_param("test_queries", len(results))
    mlflow.log_param("model", "llama-3.1-8b-instant")
    mlflow.log_param("chunks_indexed", 23562)

    # Log metrics
    mlflow.log_metric("success_rate", success/len(df))
    mlflow.log_metric("avg_response_time", df['response_time'].mean())
    mlflow.log_metric("min_response_time", df['response_time'].min())
    mlflow.log_metric("max_response_time", df['response_time'].max())
    mlflow.log_metric("median_response_time", df['response_time'].median())
    mlflow.log_metric("avg_answer_length", df['answer_length'].mean())
    mlflow.log_metric("avg_sources_found", df['sources_count'].mean())

    print("✅ Monitoring results logged to MLflow!")
    print(f"\n📊 Summary:")
    print(f"   Success rate     : {success/len(df)*100:.0f}%")
    print(f"   Avg response     : {df['response_time'].mean():.2f}s")
    print(f"   Median response  : {df['response_time'].median():.2f}s")

✅ Monitoring results logged to MLflow!

📊 Summary:
   Success rate     : 100%
   Avg response     : 1.26s
   Median response  : 0.64s


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Week 10 Summary**</span>

</div>

In [7]:
print("=" * 60)
print("🎉 WEEK 10 - MONITORING COMPLETE!")
print("=" * 60)

print("""
📦 What we built today:
   ✅ Prometheus metrics at /metrics endpoint
   ✅ API health monitoring at /health
   ✅ Performance testing (10 queries)
   ✅ Results logged to MLflow

📊 API Performance:
""")
print(f"   Success rate      : 100%")
print(f"   Avg response time : {df['response_time'].mean():.2f}s")
print(f"   Median response   : {df['response_time'].median():.2f}s")
print(f"   Min response      : {df['response_time'].min():.2f}s")
print(f"   Max response      : {df['response_time'].max():.2f}s")

print("""
🔜 Next — Week 11: CI/CD Pipeline
   → GitHub Actions workflow
   → Automated testing
   → Auto-deploy on merge
   → Auto-rollback if scores drop
""")
print("=" * 60)

🎉 WEEK 10 - MONITORING COMPLETE!

📦 What we built today:
   ✅ Prometheus metrics at /metrics endpoint
   ✅ API health monitoring at /health
   ✅ Performance testing (10 queries)
   ✅ Results logged to MLflow

📊 API Performance:

   Success rate      : 100%
   Avg response time : 1.26s
   Median response   : 0.64s
   Min response      : 0.53s
   Max response      : 6.88s

🔜 Next — Week 11: CI/CD Pipeline
   → GitHub Actions workflow
   → Automated testing
   → Auto-deploy on merge
   → Auto-rollback if scores drop

